In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
task = "Write a rejection email to a candidate. "

print("--- Lazy Prompt ---")
print(llm.invoke(task).content)

--- Lazy Prompt ---
Here are a few options for a rejection email, ranging from a more general one (after initial application) to one for a candidate who went through interviews. Choose the one that best fits your situation.

**Key Principles for Rejection Emails:**
*   **Promptness:** Send it as soon as a decision is made.
*   **Clarity:** Be clear that they haven't been selected.
*   **Professionalism:** Maintain a respectful and appreciative tone.
*   **No Specific Feedback (Generally):** Avoid giving specific reasons unless you're confident it will be constructive and won't open up debate or legal issues. "Another candidate was a closer fit" is usually sufficient.
*   **Keep it Concise:** Get to the point politely.

---

### Option 1: After Initial Application/Screening (No Interview)

**Subject: Update on Your Application for [Job Title] at [Company Name]**

Dear [Candidate Name],

Thank you for your interest in the [Job Title] position at [Company Name]. We appreciate you taking t

In [3]:
structured_prompt = """
# Context
You are the richest man on the planet. You think like a visionary billionaire 
who focuses on innovation, impact, and long-term thinking.

# Task
Define the word "Money" in one powerful sentence.

# Rules
- The sentence must be motivational.
- It must sound like business wisdom.
- Keep it under 20 words.

# Output Format
Return only the sentence. No extra text.
"""
print("--- Structured Prompt ---")
print(llm.invoke(structured_prompt).content)

--- Structured Prompt ---
Money is the catalyst for innovation, accelerating impact and shaping the future.


In [10]:
python_prompt = """
# Context
You are a Senior Python Developer.

# Objective
Write a Python function to reverse a string.

# Constraints
- Must use recursion
- Do NOT use slicing [::-1]

# Style
Include detailed docstrings.

# Output Format
Return only Python code.
"""

print(llm.invoke(python_prompt).content)


```python
def reverse_string_recursive(s: str) -> str:
    """
    Reverses a given string using a recursive approach.

    This function takes a string and returns its reversed version by
    recursively processing characters. It breaks the problem down into
    smaller sub-problems until a base case is met.

    Constraints:
    - Must use recursion.
    - Explicitly avoids the string slicing shortcut `[::-1]`. Other forms
      of slicing (e.g., `s[:-1]` to get all but the last character,
      `s[-1]` to get the last character) are considered acceptable
      as they are not the forbidden `[::-1]` shortcut and are standard
      ways to access parts of a string in Python.

    Args:
        s (str): The input string to be reversed.

    Returns:
        str: The reversed string.

    Examples:
        >>> reverse_string_recursive("hello")
        'olleh'
        >>> reverse_string_recursive("Python")
        'nohtyP'
        >>> reverse_string_recursive("")
        ''
        >>> r

In [ ]:
# Zero Shot
prompt_zero = "Combine 'Tasty' and 'Pastry' into a fancy new word."
print(f"Zero-Shot: {llm.invoke(prompt_zero).content}")

Zero-Shot: The best blend for "Tasty" and "Pastry" into a fancy new word is:

**Tastry**

It's concise, sounds elegant, and clearly combines elements of both original words.


In [6]:
# Few Shot
prompt_few = """
Combine words into a funny new word. Give a sarcastic definition.

Input: Breakfast + Lunch
Output: Brunch (An excuse to drink alcohol before noon)

Input: Chill + Relax
Output: Chillax (What annoying people say when you get panic attacks)

Input: Tried + Hungry
Output: 
"""

print(f"Few-Shot: {llm.invoke(prompt_few).content}")


Few-Shot: Output: Tungry (The profound state of being too tired to cook anything healthy, but too hungry to care about the consequences.)


In [9]:
# Dynamic Few Shotting
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {"input":"The internet is down.", "output":"We are observing connectivity latency."}, 
    {"input":"This code implies a bug", "output":"The logic suggests unintended behaviour."},
    {"input":"I hate this feature.", "output":"This feature does not align with my preferences."}
]

example_fmt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_fmt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Corpo-Speak Translator. Rewrite the input to sound professional."),
    few_shot_prompt,
    ("human", "{text}")
])

chain = final_prompt | llm
print(chain.invoke({"text": "This app sucks."}).content)


The application's current performance is suboptimal.
